# 10 · Student baseline (GPU)

Trains a student with `src/student/train.py` on one T4: the **gold-only DistilBERT baseline**
(`alpha = 1`) by default, and optionally a **mixed run** on the stack's soft labels.

* Data: the Jigsaw Toxic Comment Classification Challenge train file, cleaned with the teacher
  lane's `src/teacher/data.py` (the same text and ids the teacher labeled). It is written to
  `/tmp`, not `/kaggle/working`, so no comment text is saved as notebook output.
* Split: the harness's temporary seeded 85/15 split of raw train (`data.source: temp`), until
  the data pipeline's official splits merge. The gate (test) set is never read.
* Outputs in `/kaggle/working/`: `student/<run>/` (model, `meta.json`, validation prediction
  file, split ids, metrics), the MLflow SQLite store `mlruns/`, and `token_lengths_*.json`.

**Before the first run:**
1. Accept the rules at https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/rules,
   then **Add Input → Competitions → `jigsaw-toxic-comment-classification-challenge`**.
2. For a mixed run only (`MIXED = True`): **Add-ons → Secrets**, add `HF_TOKEN` (an HF read
   token with access to the `distillate-team` organization). Gold-only runs need no secret.

**Session settings:** Accelerator **GPU T4** (one is used), Internet **On**.
Run with **"Save Version → Save & Run All"** so the outputs are kept.

In [ ]:
# Code version. Pin a commit hash for runs you report.
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "shivam/student"

GOLD_OVERRIDES = []                          # e.g. ["train.epochs=2", "data.limit=20000"]
MIXED = False                                # also train gold + soft labels after the baseline
MIXED_OVERRIDES = ["run_name=distilbert-mixed", "loss.alpha=0.5", "loss.temperature=2.0"]

In [ ]:
import subprocess
import sys
from pathlib import Path

# iterative-stratification for the split; mlflow for tracking. torch and transformers come
# with the Kaggle image.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "mlflow"], check=True)

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet", "--ff-only"], check=False)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4"

In [ ]:
# Hugging Face token from Secrets (mixed runs only). Never printed.
import os

if MIXED:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 1. Raw train (cleaned, in /tmp)

In [ ]:
from src.teacher.config import load_config as load_teacher_config
from src.teacher.data import check_frame, find_competition_dir, read_competition_split, to_raw_frame

DATA = Path("/tmp/wiki")
DATA.mkdir(parents=True, exist_ok=True)
RAW_TRAIN = DATA / "wiki_train_raw.parquet"

teacher_cfg = load_teacher_config()
comp_dir = find_competition_dir("/kaggle/input")   # fails if the competition is not attached
raw = to_raw_frame(read_competition_split(comp_dir, "train"))
print(check_frame(raw, teacher_cfg["data"]["expected_rows"]["train"]))   # 159,571 rows
raw.to_parquet(RAW_TRAIN, index=False)
del raw

In [ ]:
# Paths for every run: outputs under /kaggle/working, downloads and text under /tmp.
OUT = Path("/kaggle/working")
PATHS = [
    f"paths.raw_train={RAW_TRAIN}",
    f"paths.out_dir={OUT / 'student'}",
    f"paths.mlflow_db={OUT / 'mlruns' / 'mlflow.db'}",
    f"paths.mlflow_artifacts={OUT / 'mlruns' / 'artifacts'}",
    f"paths.reports_dir={OUT / 'reports'}",
    "paths.cache_dir=/tmp/cache",
]

## 2. Truncation share (decides `model.max_length`)

In [ ]:
import json
import shutil

from src.student import token_lengths
from src.student.config import load_config

lengths = token_lengths.run(load_config(None, PATHS))
print(json.dumps(lengths, indent=2))
for f in (REPO_DIR / "Model-Pipeline" / "artifacts" / "student").glob("token_lengths_*.json"):
    shutil.copy(f, OUT / f.name)

## 3. Gold-only baseline (alpha = 1)

In [ ]:
from src.student import train

gold = train.run(load_config(None, PATHS + GOLD_OVERRIDES))
print(json.dumps({k: gold[k] for k in ("model_version", "best_epoch", "best_val_mean_auc",
                                       "best_val_mean_pr_auc", "duration_s")}, indent=2))

## 4. Mixed: gold + stack soft labels (optional, `MIXED = True`)

In [ ]:
runs = {"gold": gold}
if MIXED:
    runs["mixed"] = train.run(load_config(None, PATHS + MIXED_OVERRIDES))

## 5. Results (validation of the temporary split, unweighted)

In [ ]:
import pandas as pd

from src.teacher.config import LABELS

table = pd.DataFrame({
    name: {"mean_auc": s["best_val_mean_auc"], "mean_pr_auc": s["best_val_mean_pr_auc"],
           **{f"auc_{k}": s["history"][s["best_epoch"]][f"val_auc_{k}"] for k in LABELS},
           "best_epoch": s["best_epoch"], "minutes": round(s["duration_s"] / 60, 1)}
    for name, s in runs.items()})
print(table.round(4).to_string())
table.to_csv(OUT / "student_results.csv")

In [ ]:
# What is kept as output (no comment text: ids, scores, and metrics only).
for p in sorted(OUT.rglob("*")):
    if p.is_file() and "mlruns" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT)}")